In [0]:

# ============================================================
# PHASE 5 — PRODUCTION GOLD JOB
# Project: Team 3 Order Fulfilment
# Outputs:
#   1. order_line_fulfilment
#   2. order_service_performance
#   3. regional_backlog
#
# Source: Accepted Silver tables
# Write strategy: Deterministic full refresh / overwrite
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

CATALOG = "dbwteam3"
SCHEMA = "order_fulfillment"
AS_OF_TIME = "2026-10-01T00:00:00"

BASE = f"{CATALOG}.{SCHEMA}"

ORDERS_TABLE = f"{BASE}.silver_orders"
ORDER_LINES_TABLE = f"{BASE}.silver_order_lines"
DELIVERY_EVENTS_TABLE = f"{BASE}.silver_delivery_events"
PHARMACIES_TABLE = f"{BASE}.silver_pharmacies"

GOLD_LINE_TABLE = f"{BASE}.order_line_fulfilment"
GOLD_ORDER_TABLE = f"{BASE}.order_service_performance"
GOLD_REGION_TABLE = f"{BASE}.regional_backlog"

print("=" * 65)
print("PHASE 5 GOLD JOB STARTED")
print(f"Catalog/schema : {BASE}")
print(f"Reporting as-of: {AS_OF_TIME}")
print("=" * 65)


# ------------------------------------------------------------
# 2. READ AND CHECK SILVER INPUTS
# ------------------------------------------------------------

orders = spark.table(ORDERS_TABLE)
order_lines = spark.table(ORDER_LINES_TABLE)
events = spark.table(DELIVERY_EVENTS_TABLE)
pharmacies = spark.table(PHARMACIES_TABLE)

required_columns = {
    ORDERS_TABLE: {
        "order_id", "order_time", "promised_delivery_time",
        "order_status", "pharmacy_id"
    },
    ORDER_LINES_TABLE: {
        "order_id", "line_id", "ordered_quantity"
    },
    DELIVERY_EVENTS_TABLE: {
        "event_id", "order_id", "line_id",
        "delivery_time", "delivered_quantity"
    },
    PHARMACIES_TABLE: {"pharmacy_id", "region"}
}

input_dataframes = {
    ORDERS_TABLE: orders,
    ORDER_LINES_TABLE: order_lines,
    DELIVERY_EVENTS_TABLE: events,
    PHARMACIES_TABLE: pharmacies
}

for table_name, required in required_columns.items():
    actual = set(input_dataframes[table_name].columns)
    missing = required - actual

    if missing:
        raise ValueError(
            f"Missing required columns in {table_name}: "
            f"{sorted(missing)}"
        )

if orders.filter(F.col("order_id").isNull()).limit(1).count():
    raise ValueError("Silver orders contains a null order_id.")

if order_lines.filter(
    F.col("order_id").isNull()
    | F.col("line_id").isNull()
).limit(1).count():
    raise ValueError("Silver order lines contains a null business key.")

if order_lines.groupBy("order_id", "line_id").count().filter(
    F.col("count") > 1
).limit(1).count():
    raise ValueError("Duplicate order_id + line_id in Silver order lines.")

if order_lines.filter(
    F.col("ordered_quantity").isNull()
    | (F.col("ordered_quantity") <= 0)
).limit(1).count():
    raise ValueError("Invalid ordered_quantity in Silver order lines.")

if events.filter(F.col("event_id").isNull()).limit(1).count():
    raise ValueError("Silver delivery events contains a null event_id.")

if events.groupBy("event_id").count().filter(
    F.col("count") > 1
).limit(1).count():
    raise ValueError("Duplicate event_id in Silver delivery events.")

print("Silver input checks passed.")
print("Silver orders:", orders.count())
print("Silver order lines:", order_lines.count())
print("Silver delivery events:", events.count())
print("Silver pharmacies:", pharmacies.count())


# ------------------------------------------------------------
# 3. BUILD ORDER-LINE FULFILMENT
# ------------------------------------------------------------

# Aggregate events before joining to order lines. This avoids
# multiplying order-line rows when several events exist per line.

delivery_summary = (
    events
    .groupBy("order_id", "line_id")
    .agg(
        F.sum("delivered_quantity").alias(
            "uncapped_delivered_units"
        ),
        F.max("delivery_time").alias("last_delivery_time"),
        F.max(
            F.when(
                F.col("delivered_quantity") > 0,
                F.col("delivery_time")
            )
        ).alias("last_positive_delivery_time")
    )
)

# Determine the exact first event timestamp at which cumulative
# delivered quantity reaches the ordered quantity.

delivery_window = (
    Window
    .partitionBy("order_id", "line_id")
    .orderBy("delivery_time", "event_id")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

event_progress = events.withColumn(
    "cumulative_delivered_units",
    F.sum("delivered_quantity").over(delivery_window)
)

completion_times = (
    event_progress
    .join(
        order_lines.select(
            "order_id", "line_id", "ordered_quantity"
        ),
        ["order_id", "line_id"],
        "inner"
    )
    .filter(
        F.col("cumulative_delivered_units")
        >= F.col("ordered_quantity")
    )
    .groupBy("order_id", "line_id")
    .agg(
        F.min("delivery_time").alias("completion_time")
    )
)

# Combine line, delivery and order information.

line_gold = (
    order_lines
    .join(
        delivery_summary,
        ["order_id", "line_id"],
        "left"
    )
    .join(
        orders.select(
            "order_id",
            "order_time",
            "promised_delivery_time",
            "order_status"
        ),
        "order_id",
        "left"
    )
    .join(
        completion_times,
        ["order_id", "line_id"],
        "left"
    )
    .withColumn(
        "uncapped_delivered_units",
        F.coalesce(
            F.col("uncapped_delivered_units"),
            F.lit(0)
        )
    )
    .withColumn(
        "capped_delivered_units",
        F.least(
            F.col("uncapped_delivered_units"),
            F.col("ordered_quantity")
        )
    )
    .withColumn(
        "open_units",
        F.greatest(
            F.lit(0),
            F.col("ordered_quantity")
            - F.col("capped_delivered_units")
        )
    )
    .withColumn(
        "fulfilment_percentage",
        F.round(
            F.col("capped_delivered_units")
            / F.col("ordered_quantity") * 100,
            2
        )
    )
    .withColumn(
        "is_overdelivered",
        F.col("uncapped_delivered_units")
        > F.col("ordered_quantity")
    )
    .withColumn(
        "backlog_age_days",
        F.when(
            F.col("open_units") > 0,
            F.greatest(
                F.lit(0),
                F.datediff(
                    F.to_timestamp(F.lit(AS_OF_TIME)),
                    F.col("order_time")
                )
            )
        )
    )
)

# One authoritative write for the line-level Gold table.

(
    line_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_LINE_TABLE)
)

print("Written:", GOLD_LINE_TABLE)


# ------------------------------------------------------------
# 4. BUILD ORDER SERVICE PERFORMANCE
# ------------------------------------------------------------

# Compute metrics for orders that have at least one line.

line_order_metrics = (
    line_gold
    .groupBy("order_id")
    .agg(
        F.sum("ordered_quantity").alias(
            "total_ordered_units"
        ),
        F.sum("capped_delivered_units").alias(
            "total_capped_delivered_units"
        ),
        F.sum("uncapped_delivered_units").alias(
            "total_uncapped_delivered_units"
        ),
        F.sum("open_units").alias("total_open_units"),
        F.min(
            F.when(F.col("open_units") > 0, 0)
             .otherwise(1)
        ).alias("all_lines_complete"),
        F.min(
            F.when(
                (F.col("open_units") == 0)
                & (
                    F.col("completion_time")
                    <= F.col("promised_delivery_time")
                ),
                1
            ).otherwise(0)
        ).alias("all_lines_on_time"),
        F.max(
            F.col("is_overdelivered").cast("int")
        ).alias("has_overdelivery")
    )
)

# Left join ensures every Silver order is retained, including
# orders that have no order lines.

order_gold = (
    orders.select(
        "order_id",
        "promised_delivery_time",
        "order_status"
    )
    .join(
        line_order_metrics,
        "order_id",
        "left"
    )
    .withColumn(
        "total_ordered_units",
        F.coalesce(F.col("total_ordered_units"), F.lit(0))
    )
    .withColumn(
        "total_capped_delivered_units",
        F.coalesce(
            F.col("total_capped_delivered_units"), F.lit(0)
        )
    )
    .withColumn(
        "total_uncapped_delivered_units",
        F.coalesce(
            F.col("total_uncapped_delivered_units"), F.lit(0)
        )
    )
    .withColumn(
        "total_open_units",
        F.coalesce(F.col("total_open_units"), F.lit(0))
    )
    .withColumn(
        "all_lines_complete",
        F.coalesce(F.col("all_lines_complete"), F.lit(0))
    )
    .withColumn(
        "all_lines_on_time",
        F.coalesce(F.col("all_lines_on_time"), F.lit(0))
    )
    .withColumn(
        "has_overdelivery",
        F.coalesce(F.col("has_overdelivery"), F.lit(0))
    )
    .withColumn(
        "order_fulfilment_percentage",
        F.when(
            F.col("total_ordered_units") > 0,
            F.round(
                F.col("total_capped_delivered_units")
                / F.col("total_ordered_units") * 100,
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "is_cancelled",
        F.lower(F.trim(F.col("order_status"))) == "cancelled"
    )
    .withColumn(
        "is_promised_by_as_of",
        F.col("promised_delivery_time")
        <= F.to_timestamp(F.lit(AS_OF_TIME))
    )
    .withColumn(
        "otif_denominator_eligible",
        (~F.col("is_cancelled"))
        & F.col("is_promised_by_as_of")
    )
    .withColumn(
        "otif_pass",
        F.col("otif_denominator_eligible")
        & (F.col("total_ordered_units") > 0)
        & (F.col("all_lines_complete") == 1)
        & (F.col("all_lines_on_time") == 1)
    )
    .withColumn(
        "is_overdue",
        (~F.col("is_cancelled"))
        & F.col("is_promised_by_as_of")
        & (
            (F.col("total_open_units") > 0)
            | (F.col("total_ordered_units") == 0)
        )
    )
)

(
    order_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ORDER_TABLE)
)

print("Written:", GOLD_ORDER_TABLE)


# ------------------------------------------------------------
# 5. BUILD REGIONAL BACKLOG
# ------------------------------------------------------------

order_regions = (
    order_gold
    .join(
        orders.select("order_id", "pharmacy_id"),
        "order_id",
        "left"
    )
    .join(
        pharmacies.select("pharmacy_id", "region"),
        "pharmacy_id",
        "left"
    )
)

regional_gold = (
    order_regions
    .groupBy("region")
    .agg(
        F.sum("total_open_units").alias("total_open_units"),
        F.sum(
            F.when(
                F.col("total_open_units") > 0, 1
            ).otherwise(0)
        ).alias("orders_with_open_units"),
        F.sum(
            F.col("is_overdue").cast("int")
        ).alias("overdue_orders"),
        F.sum(
            F.when(
                F.col("total_ordered_units") == 0, 1
            ).otherwise(0)
        ).alias("orders_without_lines")
    )
    .withColumn(
        "as_of_time",
        F.to_timestamp(F.lit(AS_OF_TIME))
    )
)

(
    regional_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_REGION_TABLE)
)

print("Written:", GOLD_REGION_TABLE)


# ------------------------------------------------------------
# 6. VALIDATE PERSISTED GOLD TABLES
# ------------------------------------------------------------

gold_lines = spark.table(GOLD_LINE_TABLE)
gold_orders = spark.table(GOLD_ORDER_TABLE)
gold_regions = spark.table(GOLD_REGION_TABLE)

line_count = gold_lines.count()
order_count = gold_orders.count()
region_count = gold_regions.count()

expected_line_count = order_lines.count()
expected_order_count = orders.count()

duplicate_line_keys = (
    gold_lines
    .groupBy("order_id", "line_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_order_keys = (
    gold_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_open_units = gold_lines.filter(
    F.col("open_units") < 0
).count()

invalid_capped_units = gold_lines.filter(
    F.col("capped_delivered_units")
    > F.col("ordered_quantity")
).count()

missing_orders = (
    orders.select("order_id")
    .join(
        gold_orders.select("order_id"),
        "order_id",
        "left_anti"
    )
    .count()
)

cancelled_eligible_orders = gold_orders.filter(
    F.col("is_cancelled")
    & F.col("otif_denominator_eligible")
).count()

passing_incomplete_orders = gold_orders.filter(
    F.col("otif_pass")
    & (F.col("total_open_units") > 0)
).count()

passing_late_orders = (
    gold_orders.alias("o")
    .join(
        gold_lines.alias("l"),
        "order_id",
        "inner"
    )
    .filter(
        F.col("o.otif_pass")
        & (
            (F.col("l.open_units") > 0)
            | (
                F.col("l.completion_time")
                > F.col("o.promised_delivery_time")
            )
        )
    )
    .count()
)

# Cross-check totals between order-level and regional Gold.

order_totals = gold_orders.agg(
    F.sum("total_open_units").alias("open_units"),
    F.sum(F.col("is_overdue").cast("int")).alias("overdue"),
    F.sum(
        F.when(
            F.col("total_ordered_units") == 0, 1
        ).otherwise(0)
    ).alias("without_lines")
).first()

region_totals = gold_regions.agg(
    F.sum("total_open_units").alias("open_units"),
    F.sum("overdue_orders").alias("overdue"),
    F.sum("orders_without_lines").alias("without_lines")
).first()

checks = {
    "line_count_matches_silver": line_count == expected_line_count,
    "order_count_matches_silver": order_count == expected_order_count,
    "five_regional_rows": region_count == 5,
    "unique_order_line_keys": duplicate_line_keys == 0,
    "unique_order_keys": duplicate_order_keys == 0,
    "nonnegative_open_units": invalid_open_units == 0,
    "capped_delivery_not_above_ordered":
        invalid_capped_units == 0,
    "all_orders_retained": missing_orders == 0,
    "cancelled_orders_excluded_from_otif":
        cancelled_eligible_orders == 0,
    "no_incomplete_order_passes_otif":
        passing_incomplete_orders == 0,
    "no_late_line_order_passes_otif":
        passing_late_orders == 0,
    "regional_open_units_reconcile":
        order_totals["open_units"] == region_totals["open_units"],
    "regional_overdue_reconciles":
        order_totals["overdue"] == region_totals["overdue"],
    "regional_orders_without_lines_reconcile":
        order_totals["without_lines"] == region_totals["without_lines"]
}

print("\n" + "=" * 65)
print("GOLD VALIDATION RESULTS")
print("=" * 65)

for check_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check_name}")

failed_checks = [
    name for name, passed in checks.items() if not passed
]

print("\nFINAL TABLE COUNTS")
print("order_line_fulfilment:", line_count)
print("order_service_performance:", order_count)
print("regional_backlog:", region_count)

print("\nBUSINESS METRICS")
print("Total open units:", order_totals["open_units"])
print("Overdue orders:", order_totals["overdue"])
print("Orders without lines:", order_totals["without_lines"])
print(
    "OTIF eligible orders:",
    gold_orders.filter(
        F.col("otif_denominator_eligible")
    ).count()
)
print(
    "OTIF passing orders:",
    gold_orders.filter(F.col("otif_pass")).count()
)

if failed_checks:
    raise RuntimeError(
        "Gold job validation failed: "
        + ", ".join(failed_checks)
    )

print("\nPHASE 5 GOLD JOB COMPLETED SUCCESSFULLY.")


PHASE 5 GOLD JOB STARTED
Catalog/schema : dbwteam3.order_fulfillment
Reporting as-of: 2026-10-01T00:00:00
Silver input checks passed.
Silver orders: 2040
Silver order lines: 4773
Silver delivery events: 6018
Silver pharmacies: 100
Written: dbwteam3.order_fulfillment.order_line_fulfilment
Written: dbwteam3.order_fulfillment.order_service_performance
Written: dbwteam3.order_fulfillment.regional_backlog

GOLD VALIDATION RESULTS
PASS: line_count_matches_silver
PASS: order_count_matches_silver
PASS: five_regional_rows
PASS: unique_order_line_keys
PASS: unique_order_keys
PASS: nonnegative_open_units
PASS: capped_delivery_not_above_ordered
PASS: all_orders_retained
PASS: cancelled_orders_excluded_from_otif
PASS: no_incomplete_order_passes_otif
PASS: no_late_line_order_passes_otif
PASS: regional_open_units_reconcile
PASS: regional_overdue_reconciles
PASS: regional_orders_without_lines_reconcile

FINAL TABLE COUNTS
order_line_fulfilment: 4773
order_service_performance: 2040
regional_backlog: 5
